# Cartographie prédictive GWS — Ridge

Notebook Google Colab généré à partir du script Python d'origine : **`Ridge.py`**

**Dossier de données attendu sur Google Drive :** `/content/drive/MyDrive/input_data_ML`
(il doit contenir `name_files.txt` et les rasters listés dedans).

**Ordre d'exécution :** lancer les cellules de haut en bas (*Exécution → Tout exécuter*).
Chaque cellule exécute directement son étape, comme le script d'origine (qui n'a pas de fonction `main()`).
Les résultats sont écrits dans `input_data_ML/GWS_PREDICTIONS_GEE_RIDGE` sur Drive.

Le code du script est repris tel quel. Les seules lignes ajoutées ou modifiées pour Colab
sont repérées par le commentaire `# [COLAB]`.

### Description (en-tête du script d'origine)

```text
# ============================================================
#  CARTOGRAPHIE PREDICTIVE GWS
#  (TRAIN = longue période, TEST = 5% des dates, CLIP GEE, SANS SOUS-ÉCHANTILLONNAGE)
#
#  - Modèle Ridge (sklearn) sur grille GLDAS (~25 km)
#
#  CORRECTIONS IMPORTANTES (pour ton répertoire réel) :
#   - Résolution automatique des sections dans name_files.txt
#   - Précipitations: Precipitations_API_IDW (fallback Precipitations_IDW, etc.)
#   - Extraction date flexible: YYYYMMDD ou DDMMYYYY
#
#  Features:
#   [PRECIP, LST, NDVI_ext, ET_ext, Qsb, RZSM, SoilM, sinDOY, cosDOY]
# ============================================================
```

## A. Préparation de l'environnement Colab

### A.1 Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### A.2 Installer les dépendances

`rasterio` n'est pas préinstallé sur Colab (scikit-learn, numpy et joblib le sont).

In [ ]:
!pip install -q rasterio

### A.3 Imports

In [ ]:
import os
import re
import time
import bisect
from datetime import datetime, date

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import joblib

# [COLAB] Versions des bibliothèques utilisées
from importlib.metadata import version as _pkg_version
for _pkg in ('numpy', 'scikit-learn', 'rasterio'):
    print(f"{_pkg:13s}: {_pkg_version(_pkg)}")

## 1. PARAMÈTRES GÉNÉRAUX

`BASE_DIR` pointe vers le dossier `input_data_ML` de votre Drive.

In [ ]:
# ------------------------------------------------------------
# 1) PARAMÈTRES GÉNÉRAUX
# ------------------------------------------------------------
np.random.seed(42)

# [COLAB] Dossier d'entrée sur Google Drive (remplace C:\Users\PC\Downloads\input_data_ML)
BASE_DIR  = "/content/drive/MyDrive/input_data_ML"
NAME_FILE = os.path.join(BASE_DIR, "name_files.txt")

# [COLAB] Arrêt immédiat si Drive n'est pas monté ou si le dossier est mal placé
# (sinon le dossier de sortie serait créé hors de Drive et perdu en fin de session)
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        f"Dossier introuvable: {BASE_DIR} -> monter Drive (cellule A.1) et vérifier "
        "que input_data_ML est bien à la racine de 'Mon Drive'."
    )
if not os.path.isfile(NAME_FILE):
    raise FileNotFoundError(f"Fichier introuvable: {NAME_FILE}")

OUT_DIR = os.path.join(BASE_DIR, "GWS_PREDICTIONS_GEE_RIDGE")
os.makedirs(OUT_DIR, exist_ok=True)

NODATA_VALUE  = -9999.0
TEST_FRACTION = 0.05

# Fenêtres
TRAIN_START = date(2021, 1, 1)
TRAIN_END   = date(2025, 6, 30)

FORECAST_START = datetime(2025, 7, 1)
FORECAST_END   = datetime(2025, 10, 30)

# Dates exclues (LST manquante 2022-10-11 → 2022-10-22)
EXCLUDED_DATES = {date(2022, 10, d) for d in range(11, 23)}

# Hyperparamètres Ridge (ajuste si besoin)
RIDGE_ALPHA = 1.0

print("=== PARAMÈTRES ===")
print("BASE_DIR:", BASE_DIR)
print("NAME_FILE:", NAME_FILE)
print("OUT_DIR:", OUT_DIR)
print("TRAIN:", TRAIN_START, "→", TRAIN_END)
print("TEST_FRACTION:", TEST_FRACTION)
print("FORECAST:", FORECAST_START.date(), "→", FORECAST_END.date())
print("RIDGE_ALPHA:", RIDGE_ALPHA)
print("==================\n")

print("Dates exclues (LST manquante) :")
for d in sorted(EXCLUDED_DATES):
    print(" -", d)
print()

## 2. OUTILS: PARSING + SECTIONS ROBUSTES

Adaptation Colab : les entrées de `name_files.txt` écrites sous Windows
(antislashs `\`, chemins absolus `C:\...\input_data_ML\...`) sont converties
automatiquement en chemins Drive par `_win_to_colab_path`.

In [ ]:
# ------------------------------------------------------------
# 2) OUTILS: PARSING + SECTIONS ROBUSTES
# ------------------------------------------------------------
def normalize_key(s: str) -> str:
    return re.sub(r"[\s_\-]+", "", s.strip().lower())

def _win_to_colab_path(p: str) -> str:
    """
    [COLAB] Convertit une entrée écrite sous Windows en chemin utilisable sur Colab (Linux):
    - antislashs -> slashs
    - chemin absolu Windows (C:/.../input_data_ML/xxx) -> xxx (relatif à BASE_DIR)
    Les chemins déjà valides (relatifs, ou absolus sous BASE_DIR) sont inchangés.
    """
    p = str(p).strip().replace("\\", "/")
    if re.match(r"^[A-Za-z]:/", p):
        root_name = os.path.basename(os.path.normpath(BASE_DIR)).lower()
        parts = p.split("/")
        low = [x.lower() for x in parts]
        if root_name in low:
            rest = parts[low.index(root_name) + 1:]
            p = "/".join(rest) if rest else "."
    return p


def parse_name_file(txt_path):
    """Parse name_files.txt → dict {SectionName: [relative/path.tif, ...]}."""
    sections = {}
    current = None
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("[") and line.endswith("]"):
                current = line[1:-1].strip()
                sections[current] = []
            else:
                if current is None:
                    continue
                sections[current].append(_win_to_colab_path(line))  # [COLAB] chemin Windows -> Colab
    return sections

def resolve_section_name(sections: dict, primary: str, aliases=None):
    """Trouve la section existante correspondant à primary ou alias (matching tolérant)."""
    if aliases is None:
        aliases = []
    candidates = [primary] + list(aliases)

    norm_map = {normalize_key(k): k for k in sections.keys()}
    for c in candidates:
        nk = normalize_key(c)
        if nk in norm_map:
            return norm_map[nk]

    raise RuntimeError(
        f"Section introuvable pour '{primary}'. Essayé: {candidates}.\n"
        f"Sections dispo: {list(sections.keys())}"
    )

def extract_date_flexible(path):
    """
    Extrait une date depuis le nom de fichier (8 chiffres).
    Supporte YYYYMMDD ou DDMMYYYY.
    """
    name = os.path.basename(path)
    m = re.search(r"(\d{8})", name)
    if not m:
        raise ValueError(f"Impossible d'extraire une date (8 chiffres) depuis : {name}")
    s = m.group(1)

    # Tentative YYYYMMDD
    y = int(s[0:4]); mth = int(s[4:6]); d = int(s[6:8])
    if 1900 <= y <= 2100:
        try:
            return date(y, mth, d)
        except ValueError:
            pass

    # Tentative DDMMYYYY
    dd = int(s[0:2]); mm = int(s[2:4]); yy = int(s[4:8])
    try:
        return date(yy, mm, dd)
    except ValueError:
        raise ValueError(f"Date illisible dans {name}: {s}")

def build_date_to_path_dict_from_section(sections, section_name, base_dir):
    """Construit dict {date: abs_path} à partir d'une section."""
    if section_name not in sections:
        raise RuntimeError(f"Section '{section_name}' absente de {NAME_FILE}")
    out = {}
    for rel_path in sections[section_name]:
        abs_path = os.path.normpath(os.path.join(base_dir, rel_path))
        if not os.path.isfile(abs_path):
            print(f"⚠️ Fichier introuvable (ignoré) : {abs_path}")
            continue
        try:
            dk = extract_date_flexible(abs_path)
            out[dk] = abs_path
        except ValueError as e:
            print(f"⚠️ {e}")
    return out

## 3. OUTILS: RASTERS / REPROJECT / NAN

In [ ]:
# ------------------------------------------------------------
# 3) OUTILS: RASTERS / REPROJECT / NAN
# ------------------------------------------------------------
def read_raster_raw(path):
    """Lit un raster (bande 1) en float32, sans toucher NoData."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
        nodata = src.nodata
    return arr, profile, nodata

def to_float_with_nan(arr, nodata):
    """Convertit en float32 et remplace nodata -> NaN."""
    arr = arr.astype("float32")
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    return arr

def reproject_to_template(
    src_arr,
    src_profile,
    template_profile,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average,
):
    """Rééchantillonne src_arr sur la grille template_profile."""
    dst_arr = np.full(
        (template_profile["height"], template_profile["width"]),
        dst_nodata,
        dtype="float32",
    )
    kwargs = dict(
        source=src_arr,
        destination=dst_arr,
        src_transform=src_profile["transform"],
        src_crs=src_profile["crs"],
        dst_transform=template_profile["transform"],
        dst_crs=template_profile["crs"],
        dst_nodata=dst_nodata,
        resampling=resampling,
    )
    if src_nodata is not None:
        kwargs["src_nodata"] = src_nodata
    reproject(**kwargs)
    return dst_arr

def fill_nan_with_mean(arr):
    """Remplace les NaN par la moyenne du raster."""
    if np.all(np.isnan(arr)):
        return arr
    m = np.nanmean(arr)
    return np.where(np.isnan(arr), m, arr)

def build_prev_date_support(date_dict):
    """Support 'dernière date <= target_date' pour NDVI / ET."""
    dates_sorted = sorted(date_dict.keys())

    def get_prev(target_date):
        if not dates_sorted:
            return None
        idx = bisect.bisect_right(dates_sorted, target_date) - 1
        if idx < 0:
            return None
        return dates_sorted[idx]

    return get_prev

# Regions mask reconstruit (union)
REGION_MASK = None

def update_region_mask(precip_res, lst_res, ndvi_res):
    global REGION_MASK
    base_mask = ~(np.isnan(precip_res) & np.isnan(lst_res) & np.isnan(ndvi_res))
    if REGION_MASK is None:
        REGION_MASK = base_mask
        print("\n🗺️ Masque 'Regions' initial construit (1er jour).")
    else:
        REGION_MASK = REGION_MASK | base_mask

def get_region_mask(template_profile):
    global REGION_MASK
    if REGION_MASK is None:
        print("⚠️ REGION_MASK non initialisé, on prend toute la grille.")
        REGION_MASK = np.ones(
            (template_profile["height"], template_profile["width"]),
            dtype=bool,
        )
    return REGION_MASK

### Contrôle des entrées de `name_files.txt` (ajout Colab)

Vérifie, avant le calcul long, que chaque entrée du fichier se résout bien en fichier présent sur Drive.
Une section en ❌ signale un chemin à corriger. Lecture seule ; peut prendre une à deux minutes sur Drive.

In [ ]:
# [COLAB] Contrôle rapide : les entrées de name_files.txt se résolvent-elles sur Drive ?
_sections_check = parse_name_file(NAME_FILE)
print(f"{len(_sections_check)} sections lues dans {os.path.basename(NAME_FILE)}\n")

for _sec, _entries in _sections_check.items():
    _missing = [_e for _e in _entries
                if not os.path.isfile(os.path.normpath(os.path.join(BASE_DIR, _e)))]
    _n_ok = len(_entries) - len(_missing)
    _flag = "✅" if (_n_ok > 0 and not _missing) else ("⚠️" if _n_ok > 0 else "❌")
    print(f"{_flag} [{_sec}] {len(_entries)} entrées -> {_n_ok} fichiers | introuvables: {len(_missing)}")
    for _e in _missing[:3]:
        print(f"      ex. introuvable: {os.path.normpath(os.path.join(BASE_DIR, _e))}")

## 4. INDEXATION DES FICHIERS (via name_files.txt)

In [ ]:
# ------------------------------------------------------------
# 4) INDEXATION DES FICHIERS (via name_files.txt)
# ------------------------------------------------------------
print("=== INDEXATION DES SECTIONS ===")
sections = parse_name_file(NAME_FILE)
print("Sections dispo:", list(sections.keys()), "\n")

# Résolution robuste des sections (IMPORTANT pour ton cas)
SEC_PRECIP = resolve_section_name(
    sections,
    "Precipitations_API_IDW",
    aliases=["Precipitations_IDW", "Precipitations_API", "CHIRPS_Clipped", "CHIRPS", "CHIRPS_Clipped_10km"]
)
SEC_LST   = resolve_section_name(sections, "MODIS_LST_Clipped",  aliases=["LST", "MODIS_LST"])
SEC_NDVI  = resolve_section_name(sections, "MODIS_NDVI_Clipped", aliases=["NDVI", "MODIS_NDVI"])
SEC_ET    = resolve_section_name(sections, "MODIS_ET_Clipped",   aliases=["ET", "MODIS_ET"])
SEC_GWS   = resolve_section_name(sections, "GLDAS_GWS_Clipped",  aliases=["GWS", "GLDAS_GWS"])
SEC_QSB   = resolve_section_name(sections, "ERA5_Qsb_Clipped",   aliases=["Qsb", "ERA5_QSB"])
SEC_RZSM  = resolve_section_name(sections, "SMAP_RZSM_Clipped",  aliases=["RZSM", "SMAP_RZSM"])
SEC_SOILM = resolve_section_name(sections, "SMAP_SoilM_Clipped", aliases=["SoilM", "SMAP_SoilM"])

print("✅ Sections utilisées:")
print(" - PRECIP :", SEC_PRECIP)
print(" - LST    :", SEC_LST)
print(" - NDVI   :", SEC_NDVI)
print(" - ET     :", SEC_ET)
print(" - GWS    :", SEC_GWS)
print(" - Qsb    :", SEC_QSB)
print(" - RZSM   :", SEC_RZSM)
print(" - SoilM  :", SEC_SOILM)
print()

precip_dict = build_date_to_path_dict_from_section(sections, SEC_PRECIP, BASE_DIR)
lst_dict    = build_date_to_path_dict_from_section(sections, SEC_LST,    BASE_DIR)
ndvi_dict   = build_date_to_path_dict_from_section(sections, SEC_NDVI,   BASE_DIR)
et_dict     = build_date_to_path_dict_from_section(sections, SEC_ET,     BASE_DIR)
gws_dict    = build_date_to_path_dict_from_section(sections, SEC_GWS,    BASE_DIR)
qsb_dict    = build_date_to_path_dict_from_section(sections, SEC_QSB,    BASE_DIR)
rzsm_dict   = build_date_to_path_dict_from_section(sections, SEC_RZSM,   BASE_DIR)
soilm_dict  = build_date_to_path_dict_from_section(sections, SEC_SOILM,  BASE_DIR)

print("Nb dates PRECIP :", len(precip_dict))
print("Nb dates LST    :", len(lst_dict))
print("Nb dates NDVI   :", len(ndvi_dict))
print("Nb dates ET     :", len(et_dict))
print("Nb dates GWS    :", len(gws_dict))
print("Nb dates Qsb    :", len(qsb_dict))
print("Nb dates RZSM   :", len(rzsm_dict))
print("Nb dates SoilM  :", len(soilm_dict))
print()

get_ndvi_date_for = build_prev_date_support(ndvi_dict)
get_et_date_for   = build_prev_date_support(et_dict)

## 5. DATES COMMUNES + SPLIT TRAIN/TEST

In [ ]:
# ------------------------------------------------------------
# 5) DATES COMMUNES + SPLIT TRAIN/TEST
# ------------------------------------------------------------
print("=== CONSTRUCTION DES DATES TRAIN / TEST ===")

all_common_dates = sorted(
    set(precip_dict.keys())
    & set(lst_dict.keys())
    & set(gws_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

print("Nombre total de dates communes (PRECIP ∩ LST ∩ GWS ∩ Qsb ∩ RZSM ∩ SoilM) :", len(all_common_dates))
if not all_common_dates:
    raise RuntimeError("⚠️ Aucune date commune entre les variables journalières.")

date_candidates = [
    d for d in all_common_dates
    if (TRAIN_START <= d <= TRAIN_END) and (d not in EXCLUDED_DATES)
]
if not date_candidates:
    raise RuntimeError("⚠️ Aucune date candidate dans la fenêtre TRAIN (après exclusion).")

train_dates, test_dates = train_test_split(
    date_candidates,
    test_size=TEST_FRACTION,
    random_state=42,
    shuffle=True,
)
train_dates = sorted(train_dates)
test_dates  = sorted(test_dates)

print(f"TRAIN pool: {TRAIN_START} → {TRAIN_END}")
print(" - #train_dates:", len(train_dates), "|", min(train_dates), "→", max(train_dates))
print("TEST (5% dates):")
print(" - #test_dates :", len(test_dates),  "|", min(test_dates),  "→", max(test_dates))
print()

## 6. CONSTRUCTION DATASET (pixels)

In [ ]:
# ------------------------------------------------------------
# 6) CONSTRUCTION DATASET (pixels)
# ------------------------------------------------------------
print("=== CONSTRUCTION DU DATASET (pixels) ===")

X_train_list, y_train_list = [], []
X_test_list,  y_test_list  = [], []
template_profile = None

def build_Xy_for_date(date_key, template_profile, role="train"):
    ndvi_date = get_ndvi_date_for(date_key)
    if ndvi_date is None:
        print(f"⏭️ [{role}] {date_key}: NDVI indisponible (<= date), ignoré.")
        return None, None, template_profile

    et_date = get_et_date_for(date_key)
    if et_date is None:
        print(f"⏭️ [{role}] {date_key}: ET indisponible (<= date), ignoré.")
        return None, None, template_profile

    if date_key not in gws_dict:
        print(f"⏭️ [{role}] {date_key}: GWS manquant, ignoré.")
        return None, None, template_profile

    # Cible -> template GLDAS
    gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    if template_profile is None:
        template_profile = gws_prof
        print("\nGrille GLDAS (template) :")
        print(" - CRS  :", template_profile["crs"])
        print(" - Taille :", template_profile["width"], "x", template_profile["height"])
        print(" - Résolution approx (deg) :", template_profile["transform"][0])
        print()

    # Variables journalières obligatoires
    if any(date_key not in dct for dct in [precip_dict, lst_dict, qsb_dict, rzsm_dict, soilm_dict]):
        print(f"⏭️ [{role}] {date_key}: variable journalière manquante, ignoré.")
        return None, None, template_profile

    # Lecture entrées
    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    # Reprojection sur grille GLDAS
    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata)
    lst_res    = reproject_to_template(lst_raw,    lst_prof,    template_profile, src_nodata=lst_nodata)
    ndvi_res   = reproject_to_template(ndvi_raw,   ndvi_prof,   template_profile, src_nodata=ndvi_nodata)
    et_res     = reproject_to_template(et_raw,     et_prof,     template_profile, src_nodata=et_nodata)
    qsb_res    = reproject_to_template(qsb_raw,    qsb_prof,    template_profile, src_nodata=qsb_nodata)
    rzsm_res   = reproject_to_template(rzsm_raw,   rzsm_prof,   template_profile, src_nodata=rzsm_nodata)
    soilm_res  = reproject_to_template(soilm_raw,  soilm_prof,  template_profile, src_nodata=soilm_nodata)

    # Regions (reconstruit)
    update_region_mask(precip_res, lst_res, ndvi_res)
    region_mask = get_region_mask(template_profile)

    # Remplir NaN -> pas de trous dans les features
    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    # Pixels valides = GWS défini & dans Regions
    mask_valid = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_valid):
        print(f"⏭️ [{role}] {date_key}: aucun pixel valide, ignoré.")
        return None, None, template_profile

    # sin/cos DOY
    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2*np.pi*doy/365.0)
    cos_doy = np.cos(2*np.pi*doy/365.0)

    n = np.count_nonzero(mask_valid)
    sin_plane = np.full(n, sin_doy, dtype="float32")
    cos_plane = np.full(n, cos_doy, dtype="float32")

    X_day = np.stack([
        precip_f[mask_valid],
        lst_f[mask_valid],
        ndvi_f[mask_valid],
        et_f[mask_valid],
        qsb_f[mask_valid],
        rzsm_f[mask_valid],
        soilm_f[mask_valid],
        sin_plane,
        cos_plane
    ], axis=1)

    y_day = gws_arr[mask_valid]

    row_mask = (~np.isnan(y_day)) & (~np.isnan(X_day).any(axis=1))
    if not np.any(row_mask):
        print(f"⏭️ [{role}] {date_key}: lignes invalides (NaN), ignoré.")
        return None, None, template_profile

    X_day = X_day[row_mask]
    y_day = y_day[row_mask]

    print(f" ✔️ [{role}] {date_key}: {X_day.shape[0]} pixels.")
    return X_day, y_day, template_profile

# Build TRAIN pixels
for d in train_dates:
    Xd, yd, template_profile = build_Xy_for_date(d, template_profile, role="train")
    if Xd is None:
        continue
    X_train_list.append(Xd)
    y_train_list.append(yd)

# Build TEST pixels
for d in test_dates:
    Xd, yd, template_profile = build_Xy_for_date(d, template_profile, role="test")
    if Xd is None:
        continue
    X_test_list.append(Xd)
    y_test_list.append(yd)

if not X_train_list:
    raise RuntimeError("⚠️ Aucun pixel d'entraînement valide.")
if not X_test_list:
    raise RuntimeError("⚠️ Aucun pixel de test valide.")

X_train = np.vstack(X_train_list).astype("float32")
y_train = np.concatenate(y_train_list).astype("float32")
X_test  = np.vstack(X_test_list).astype("float32")
y_test  = np.concatenate(y_test_list).astype("float32")

print("\nDataset construit:")
print(" - X_train:", X_train.shape, "| y_train:", y_train.shape)
print(" - X_test :", X_test.shape,  "| y_test :", y_test.shape)
print("Exemple feature:", X_train[0])
print()

## 7. ENTRAINEMENT RIDGE

In [ ]:
# ------------------------------------------------------------
# 7) ENTRAINEMENT RIDGE
# ------------------------------------------------------------
print("=== ENTRAINEMENT RIDGE ===")

# split interne pixels (val)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, shuffle=True
)
print("Split interne:", X_tr.shape, X_val.shape)

# Pipeline: StandardScaler + Ridge (plus stable)
ridge = make_pipeline(
    StandardScaler(),
    Ridge(alpha=RIDGE_ALPHA, max_iter=10000, tol=1e-4)
)

t0 = time.time()
ridge.fit(X_tr, y_tr)
t1 = time.time()
training_time_ms = (t1 - t0) * 1000.0

def metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return r2, mae, rmse

r2_tr,  mae_tr,  rmse_tr  = metrics(y_tr,  ridge.predict(X_tr))
r2_val, mae_val, rmse_val = metrics(y_val, ridge.predict(X_val))
r2_te,  mae_te,  rmse_te  = metrics(y_test, ridge.predict(X_test))

print("\n╔════════════ RIDGE ════════════╗")
print(f" ➤ Temps entraînement : {training_time_ms:.2f} ms")
print(" --- TRAIN (X_tr) ---")
print(f"    R²={r2_tr:.4f} | MAE={mae_tr:.4f} | RMSE={rmse_tr:.4f}")
print(" --- VAL (X_val) ---")
print(f"    R²={r2_val:.4f} | MAE={mae_val:.4f} | RMSE={rmse_val:.4f}")
print(" --- TEST (5% dates) ---")
print(f"    R²={r2_te:.4f} | MAE={mae_te:.4f} | RMSE={rmse_te:.4f}")
print("╚═══════════════════════════════╝\n")

# Save model
ridge_path = os.path.join(OUT_DIR, "Ridge_GWS_model.joblib")
joblib.dump(ridge, ridge_path)
print("💾 Modèle sauvegardé:", ridge_path, "\n")

## 8. EVALUATION CARTOGRAPHIQUE (dates TEST)

In [ ]:
# ------------------------------------------------------------
# 8) EVALUATION CARTOGRAPHIQUE (dates TEST)
# ------------------------------------------------------------
print("=== EVALUATION CARTOGRAPHIQUE (TEST DATES) ===")

region_mask = get_region_mask(template_profile)

out_profile_test = template_profile.copy()
out_profile_test.update(dtype="float32", count=1, nodata=NODATA_VALUE)

all_true, all_pred = [], []

for date_key in test_dates:
    print(f"\n🧪 Date test: {date_key}")

    if date_key in EXCLUDED_DATES:
        print(" ⏭️ Date exclue (LST manquante).")
        continue

    ndvi_date = get_ndvi_date_for(date_key)
    et_date   = get_et_date_for(date_key)
    if ndvi_date is None or et_date is None:
        print(" ⏭️ NDVI/ET indisponible, ignoré.")
        continue
    if date_key not in gws_dict:
        print(" ⏭️ GWS manquant, ignoré.")
        continue
    if any(date_key not in dct for dct in [precip_dict, lst_dict, qsb_dict, rzsm_dict, soilm_dict]):
        print(" ⏭️ Variable journalière manquante, ignoré.")
        continue

    # vérité terrain
    gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    # entrées
    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata)
    lst_res    = reproject_to_template(lst_raw,    lst_prof,    template_profile, src_nodata=lst_nodata)
    ndvi_res   = reproject_to_template(ndvi_raw,   ndvi_prof,   template_profile, src_nodata=ndvi_nodata)
    et_res     = reproject_to_template(et_raw,     et_prof,     template_profile, src_nodata=et_nodata)
    qsb_res    = reproject_to_template(qsb_raw,    qsb_prof,    template_profile, src_nodata=qsb_nodata)
    rzsm_res   = reproject_to_template(rzsm_raw,   rzsm_prof,   template_profile, src_nodata=rzsm_nodata)
    soilm_res  = reproject_to_template(soilm_raw,  soilm_prof,  template_profile, src_nodata=soilm_nodata)

    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2*np.pi*doy/365.0)
    cos_doy = np.cos(2*np.pi*doy/365.0)

    n_pix = precip_f.size
    sin_plane = np.full(n_pix, sin_doy, dtype="float32")
    cos_plane = np.full(n_pix, cos_doy, dtype="float32")

    X_all = np.stack([
        precip_f.ravel(),
        lst_f.ravel(),
        ndvi_f.ravel(),
        et_f.ravel(),
        qsb_f.ravel(),
        rzsm_f.ravel(),
        soilm_f.ravel(),
        sin_plane,
        cos_plane
    ], axis=1)

    if np.isnan(X_all).any():
        print(" ⏭️ X_all contient encore NaN, ignoré.")
        continue

    y_pred_all = ridge.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    mask_gws = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_gws):
        print(" ⚠️ Aucun pixel GWS valide dans Regions.")
        continue

    y_true = gws_arr[mask_gws]
    y_pred = pred_clip[mask_gws]

    r2_d, mae_d, rmse_d = metrics(y_true, y_pred)
    print(f" ➤ R²={r2_d:.4f} | MAE={mae_d:.4f} | RMSE={rmse_d:.4f}")

    all_true.append(y_true)
    all_pred.append(y_pred)

    out_name = os.path.join(OUT_DIR, f"GWS_TEST_PRED_RIDGE_{date_key.strftime('%Y%m%d')}.tif")
    with rasterio.open(out_name, "w", **out_profile_test) as dst:
        dst.write(pred_clip, 1)
    print(" ✔️ Sauvé:", out_name)

if all_true:
    yT = np.concatenate(all_true)
    yP = np.concatenate(all_pred)
    r2_g, mae_g, rmse_g = metrics(yT, yP)
    print("\n📊 METRIQUES GLOBALES TEST (cartes clipées Regions)")
    print(f" ➤ R²={r2_g:.4f} | MAE={mae_g:.4f} | RMSE={rmse_g:.4f}")
else:
    print("\n⚠️ Pas de métriques globales (aucune date test valide).")

## 9. FORECAST (cartes prédictives)

In [ ]:
# ------------------------------------------------------------
# 9) FORECAST (cartes prédictives)
# ------------------------------------------------------------
print("\n=== FORECAST (PREDICTIONS CARTOGRAPHIQUES) ===")

if template_profile is None:
    raise RuntimeError("Template GLDAS introuvable (aucun GWS lu).")

out_profile = template_profile.copy()
out_profile.update(dtype="float32", count=1, nodata=NODATA_VALUE)

all_forecast_dates = sorted(
    set(precip_dict.keys())
    & set(lst_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

if not all_forecast_dates:
    print("⚠️ Aucune date commune pour forecast (journaliers).")
else:
    print("Plage possible forecast:", min(all_forecast_dates), "→", max(all_forecast_dates))

user_start = FORECAST_START.date()
user_end   = FORECAST_END.date()

if all_forecast_dates:
    forecast_start_date = max(user_start, min(all_forecast_dates))
    forecast_end_date   = min(user_end,   max(all_forecast_dates))
else:
    forecast_start_date = user_start
    forecast_end_date   = user_end

print("Fenêtre demandée:", user_start, "→", user_end)
print("Fenêtre utilisée:", forecast_start_date, "→", forecast_end_date)

forecast_dates = [
    d for d in all_forecast_dates
    if (forecast_start_date <= d <= forecast_end_date) and (d not in EXCLUDED_DATES)
]

print("Nb dates forecast:", len(forecast_dates))
if forecast_dates:
    print(" - Première:", min(forecast_dates))
    print(" - Dernière:", max(forecast_dates))

region_mask = get_region_mask(template_profile)

for date_key in forecast_dates:
    print(f"\n🛰️ Forecast date: {date_key}")

    ndvi_date = get_ndvi_date_for(date_key)
    et_date   = get_et_date_for(date_key)
    if ndvi_date is None or et_date is None:
        print(" ⏭️ NDVI/ET indisponible, ignoré.")
        continue

    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata)
    lst_res    = reproject_to_template(lst_raw,    lst_prof,    template_profile, src_nodata=lst_nodata)
    ndvi_res   = reproject_to_template(ndvi_raw,   ndvi_prof,   template_profile, src_nodata=ndvi_nodata)
    et_res     = reproject_to_template(et_raw,     et_prof,     template_profile, src_nodata=et_nodata)
    qsb_res    = reproject_to_template(qsb_raw,    qsb_prof,    template_profile, src_nodata=qsb_nodata)
    rzsm_res   = reproject_to_template(rzsm_raw,   rzsm_prof,   template_profile, src_nodata=rzsm_nodata)
    soilm_res  = reproject_to_template(soilm_raw,  soilm_prof,  template_profile, src_nodata=soilm_nodata)

    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2*np.pi*doy/365.0)
    cos_doy = np.cos(2*np.pi*doy/365.0)

    n_pix = precip_f.size
    sin_plane = np.full(n_pix, sin_doy, dtype="float32")
    cos_plane = np.full(n_pix, cos_doy, dtype="float32")

    X_all = np.stack([
        precip_f.ravel(),
        lst_f.ravel(),
        ndvi_f.ravel(),
        et_f.ravel(),
        qsb_f.ravel(),
        rzsm_f.ravel(),
        soilm_f.ravel(),
        sin_plane,
        cos_plane
    ], axis=1)

    if np.isnan(X_all).any():
        print(" ⏭️ X_all contient encore NaN, ignoré.")
        continue

    y_pred_all = ridge.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    out_name = os.path.join(OUT_DIR, f"GWS_PRED_RIDGE_{date_key.strftime('%Y%m%d')}.tif")
    with rasterio.open(out_name, "w", **out_profile) as dst:
        dst.write(pred_clip, 1)
    print(" ✔️ Sauvé:", out_name)

tifs = [f for f in os.listdir(OUT_DIR) if f.lower().endswith(".tif")]
print(f"\n📂 Total .tif générés dans {OUT_DIR} : {len(tifs)}")

## Récapitulatif des sorties (ajout Colab)

In [ ]:
# [COLAB] Nombre de fichiers présents dans le dossier de sortie, par extension
from collections import Counter

_count = Counter(os.path.splitext(_f)[1].lower() or "(sans extension)" for _f in os.listdir(OUT_DIR))
for _ext, _n in sorted(_count.items()):
    print(f"{_n:6d} fichier(s) {_ext}")
print("Dossier de sortie :", OUT_DIR)